<a href="https://colab.research.google.com/github/jhajagos/SupportingConceptSetGeneration/blob/main/UMLS_API_with_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Environment

In [14]:
import requests
import pprint
import json

In [15]:
!pip install -q -U google-generativeai

In [16]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [17]:
UMLS_API_KEY = userdata.get('UMLS_API_KEY')
UMLS_BASE_URL= " https://uts-ws.nlm.nih.gov/rest"

## UMLS API

In [18]:
t = """/search/{version}|Retrieves CUIs when searching by term or code
  /content/{version}/CUI/{CUI}|Retrieves information about a known CUI
  /content/{version}/CUI/{CUI}/atoms|Retrieves atoms and information about atoms for a known CUI
  /content/{version}/CUI/{CUI}/definitions|Retrieves definitions for a known CUI
  /content/{version}/CUI/{CUI}/relations|Retrieves NLM-asserted relationships for a known CUI
  /content/{version}/source/{source}/{id}|Retrieves information about a known source-asserted identifier
  /content/{version}/source/{source}/{id}/atoms|Retrieves information about atoms for a known source-asserted identifier
  /content/{version}/source/{source}/{id}/parents|Retrieves immediate parents of a source-asserted identifier
  /content/{version}/source/{source}/{id}/children|Retrieves immediate children of a source-asserted identifier
  /content/{version}/source/{source}/{id}/ancestors|Retrieves all ancestors of a source-asserted identifier
  /content/{version}/source/{source}/{id}/descendants|Retrieves all descendants of a source-asserted identifier
  /content/{version}/source/{source}/{id}/relations|Retrieves all relationships of a source-asserted identifier
  /content/{version}/source/{source}/{id}/attributes|Retrieves information about source-asserted attributes
  /semantic-network/{version}/TUI/{id}|Retrieves information for a known Semantic Type identifier (TUI)
  /crosswalk/{version}/source/{source}/{id}|Retrieves all source-asserted identifiers that share a UMLS CUI with a particular code"""
api_templates = [l.strip().split("|") for l in t.split("\n")]

In [19]:
import pandas as pd
pd.DataFrame(api_templates, columns=["Path", "Description"])

,Path,Description
0,/search/{version},Retrieves CUIs when searching by term or code
1,/content/{version}/CUI/{CUI},Retrieves information about a known CUI
2,/content/{version}/CUI/{CUI}/atoms,Retrieves atoms and information about atoms fo...
3,/content/{version}/CUI/{CUI}/definitions,Retrieves definitions for a known CUI
4,/content/{version}/CUI/{CUI}/relations,Retrieves NLM-asserted relationships for a kno...
5,/content/{version}/source/{source}/{id},Retrieves information about a known source-ass...
6,/content/{version}/source/{source}/{id}/atoms,Retrieves information about atoms for a known ...
7,/content/{version}/source/{source}/{id}/parents,Retrieves immediate parents of a source-assert...
8,/content/{version}/source/{source}/{id}/children,Retrieves immediate children of a source-asser...
9,/content/{version}/source/{source}/{id}/ancestors,Retrieves all ancestors of a source-asserted i...


In [20]:
import time
import logging

logging.basicConfig(level=logging.INFO)

def process_relationships(relationships):
  """Build a list of relationships from the API response"""
  relationship_list = []
  for relation in relationships:
    relationship_list += [{"relationship_source": relation['rootSource'],
                            "relationship_type": relation['relationLabel'],
                            "relationship_type_description": relation['additionalRelationLabel'],
                            "relationship_target": relation['relatedIdName'],
                            "relationship_target_code": relation["relatedId"].split("/")[-1]
                          }]
  return relationship_list

def process_hierarchial_term(terms):
  result_list = []
  for term in terms:
    result_list += [{
        "id": term["ui"],
        "name": term["name"]}
    ]
  return result_list


def umls_request(url, api_key=UMLS_API_KEY, additional_params={}):
  """Make a request to the UMLS API and return the response"""
  start_time = time.time()

  params = {"apiKey": api_key}
  params.update(additional_params)

  logging.info(f"Making request to {url} with params {params}")
  response = requests.get(url, params=params).json()

  end_time = time.time()
  logging.info(f"Request took {end_time - start_time} seconds")

  return response


def get_vocabularies():
  rl = umls_request(UMLS_BASE_URL + "/metadata/current/sources")
  return {r["abbreviation"]: r["expandedForm"] for r in rl['result']}


def get_vocabulary_languages():
  rl = umls_request(UMLS_BASE_URL + "/metadata/current/sources")
  return {r["abbreviation"]: r["language"]["expandedForm"] for r in rl['result']}


def get_code_source_information(code, source="ICD10CM", version = "current"):
  """For given terms in a source vocabulary gets context around the term and
    returns results as a dictionary."""


  languages = get_vocabulary_languages() # Get language

  r_obj = umls_request(UMLS_BASE_URL + f"/content/{version}/source/{source}/{code}")

  if "result" not in r_obj:
    return None
  else:
    r = r_obj["result"]

    name = r["name"]

    attributes_dict = {} # Gets a term attributes (MRSAT table)
    if r["attributes"] == "NONE":
      pass
    else:
      attributes = umls_request(r["attributes"], additional_params={"pageSize": 100}) # TODO: Add paging
      if "result" in attributes:
        for attribute in attributes["result"]:
          attributes_dict[attribute["name"]] = attribute["value"]

    relationship_list = [] # Gets term relationships (MRREL)
    if r["relations"] == "NONE":
      pass
    else:
      relationships = umls_request(r["relations"], additional_params={"pageSize": 100})
      if "pageCount" in relationships:
        relationship_list = process_relationships(relationships["result"])
        if relationships["pageCount"] > 1:
          for i in range(2,relationships["pageCount"]+1):
            i_relationships = umls_request(r["relations"], additional_params={"pageSize": 100, "pageNumber": i})
            relationship_list += process_relationships(i_relationships["result"])

    # Get CUIs associated with the term (MRCONSO)
    concept_url = r["concepts"]
    concepts_obj = umls_request(concept_url)

    # Get parent and children terms

    parent_list = []
    children_list = []

    parents = r["parents"]
    children = r["children"]

    if parents != "NONE":
      parent_obj = umls_request(parents)
      parent_list = process_hierarchial_term(parent_obj["result"])

    if children != "NONE":
      children_obj = umls_request(children)
      children_list = process_hierarchial_term(children_obj["result"])

    ancestors_list = []
    descendants_list = []

    ancestors = r["ancestors"]
    descendants = r["descendants"]

    if ancestors != "NONE":
      ancestors_obj = umls_request(ancestors)
      ancestors_list = process_hierarchial_term(ancestors_obj["result"])

    if descendants != "NONE":
      descendants_obj = umls_request(descendants)
      descendants_list = process_hierarchial_term(descendants_obj["result"])

    concept_dict = {}
    if "result" in concepts_obj:
      concepts = concepts_obj["result"]["results"]

      for concept in concepts:
        concept_dict[concept["ui"]] = {"concept_uri": concept["uri"]}

      # Get defintions (include only English defintions) MRDEF
      for cui in concept_dict:
        concept_obj = umls_request(concept_dict[cui]["concept_uri"])

        if "result" in concept_obj:
          cui_concept_obj = concept_obj["result"]
          semantic_types = [s["name"] for s in cui_concept_obj["semanticTypes"]]
          if len(semantic_types) == 1:
            concept_dict[cui]["semantic_type"] = semantic_types[0]
          else:
            concept_dict[cui]["semantic_type"] = semantic_types

          concept_dict[cui]["definitions"] = {}
          if "definitions" in cui_concept_obj:

            if cui_concept_obj["definitions"] != "NONE":
              defintions_obj = umls_request(cui_concept_obj["definitions"])
              for result in defintions_obj["result"]:
                vocabulary = result["rootSource"]
                if languages[vocabulary] == "English":
                  concept_dict[cui]["definitions"][vocabulary] = result["value"]

  return {"code": code, "name": name, "vocabulary": source, "concepts": concept_dict,
          "attributes": attributes_dict, "relationships": relationship_list,
          "parents": parent_list, "children": children_list,
          "ancesotors": ancestors_list, "descendants": descendants_list}

In [21]:
vocabularies = get_vocabularies()
vocabularies

{'AIR': 'AI/RHEUM, 1993',
 'CST': 'COSTART, 1995',
 'DXP': 'DXplain, 1994',
 'LCH': 'Library of Congress Subject Headings, 1990',
 'MCM': 'McMaster University Epidemiology Terms, 1992',
 'SNM': 'SNOMED-2, 2',
 'SNMI': 'SNOMED International, 1998',
 'WHO': 'WHO Adverse Reaction Terminology, 1997',
 'ULT': 'UltraSTAR, 1993',
 'ICD10': 'ICD10, 1998',
 'ICPC': 'International Classification of Primary Care, 1993',
 'QMR': 'Quick Medical Reference (QMR), 1996',
 'RCD': 'Clinical Terms Version 3 (CTV3) (Read Codes), 1999',
 'PPAC': 'Pharmacy Practice Activity Classification, 1998',
 'AOD': 'Alcohol and Other Drug Thesaurus, 2000',
 'BI': 'Beth Israel Vocabulary, 1.0',
 'RCDAE': 'Read thesaurus, American English Equivalents, 1999',
 'RCDSA': 'Read thesaurus Americanized Synthesized Terms, 1999',
 'RCDSY': 'Read thesaurus, Synthesized Terms, 1999',
 'ICD10AE': 'ICD10, American English Equivalents, 1998',
 'DMDICD10': 'German translation of ICD10, 1995',
 'DMDUMD': 'German translation of UMDNS, 

In [22]:
%%time
get_code_source_information("J00", "ICD10CM")

CPU times: user 74.3 ms, sys: 11.4 ms, total: 85.7 ms
Wall time: 1.76 s


{'code': 'J00',
 'name': 'Acute nasopharyngitis [common cold]',
 'vocabulary': 'ICD10CM',
 'concepts': {'C0009443': {'concept_uri': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0009443',
   'semantic_type': 'Disease or Syndrome',
   'definitions': {'CSP': 'catarrhal disorder of the upper respiratory tract, which may be viral or a mixed infection; marked by acute coryza, slight rise in temperature, chilly sensations, and general indisposition.',
    'MEDLINEPLUS': '<h3>What is the common cold?</h3> <p>The common cold is a mild infection of your upper respiratory tract (which includes your nose and throat). Colds are probably the most common illness. Adults have an average of 2-3 colds per year, and children have even more. Colds are more common in the winter and spring, but you can get them at any time.</p> <h3>What causes the common cold?</h3> <p>More than 200 different viruses can cause a cold, but rhinoviruses are the most common type. The viruses that cause colds are very co

In [23]:
#list(genai.list_models())

## Helper CSV tables

In [24]:
import pandas as pd
ccsr_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/ccsr_codes.csv")
ccsr_df

,code,code_type,description
0,BLD001,CCSR_ICD10CM,Nutritional anemia
1,BLD002,CCSR_ICD10CM,Hemolytic anemia
2,BLD003,CCSR_ICD10CM,Aplastic anemia
3,BLD004,CCSR_ICD10CM,Acute posthemorrhagic anemia
4,BLD005,CCSR_ICD10CM,Sickle cell trait/anemia
...,...,...,...
549,SYM016,CCSR_ICD10CM,Other general signs and symptoms
550,SYM017,CCSR_ICD10CM,Abnormal findings without diagnosis
551,SYM018,CCSR_ICD10CM,Prediabetes
552,XXX000,CCSR_ICD10CM,Unacceptable PDX


In [191]:
full_icd10cm_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/umls_ohdsi_icd10_pt.csv")
full_icd10cm_df[full_icd10cm_df["TTY"] == "HT"]

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,...,SRL,SUPPRESS,CVF,dummy,STYs,PREFIX_CODE,ICD10CMCode,concept_code,concept_id,domain_id
5,C0002438,ENG,P,L0002438,PF,S0012920,N,A17850106,NaN,NaN,...,4,N,256.0,NaN,['Disease or Syndrome'],A06,A06,A06,1567245,Condition
11,C0002962,ENG,P,L0002962,VC,S0355458,N,A17826307,NaN,NaN,...,4,N,256.0,NaN,['Sign or Symptom'],I20,I20,I20,1569125,Condition
17,C0003705,ENG,S,L0823983,PF,S1047525,N,A17835348,NaN,NaN,...,4,N,256.0,NaN,['Injury or Poisoning'],T63,T633,T63.3,1575080,Condition
18,C0003742,ENG,P,L0003742,VC,S0356512,N,A17838687,NaN,NaN,...,4,N,256.0,NaN,['Disease or Syndrome'],H18,H1841,H18.41,45600876,Condition
19,C0003803,ENG,S,L0078983,VC,S0470971,N,A17815974,NaN,NaN,...,4,N,256.0,NaN,['Congenital Abnormality'],Q07,Q070,Q07.0,1572052,Condition
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97516,C5819435,ENG,P,L18773173,VC,S22498362,Y,A35632235,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],W44,W44E4,W44.E4,786274,Condition
97535,C5889707,ENG,S,L19377796,PF,S23151648,Y,A36508591,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],T45,T45A,T45.A,1102781,Observation
97556,C5926696,ENG,P,L19377687,VC,S23151191,Y,A36508788,NaN,NaN,...,4,N,256.0,NaN,['Acquired Abnormality'],K60,K6031,K60.31,1102697,Condition
97569,C5926771,ENG,P,L19377856,VC,S23151635,Y,A36508811,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],T45,T45AX3,T45.AX3,1102791,Condition


In [176]:
ht_range_terms_icd10cm_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/ht_range_terms_icd10cm.csv")
ht_range_terms_icd10cm_df

,CUI,CODE,STR
0,C0178238,A00-A09,Intestinal infectious diseases (A00-A09)
1,C0694449,A00-B99,Certain infectious and parasitic diseases (A00...
2,C0041296,A15-A19,Tuberculosis (A15-A19)
3,C0348110,A20-A28,Certain zoonotic bacterial diseases (A20-A28)
4,C2939130,A30-A49,Other bacterial diseases (A30-A49)
...,...,...,...
313,C0582114,Z66-Z66,Do not resuscitate status (Z66)
314,C2911644,Z67-Z67,Blood type (Z67)
315,C2240399,Z68-Z68,Body mass index [BMI] (Z68)
316,C0178343,Z69-Z76,Persons encountering health services in other ...


In [177]:
prefix_codes_icd10cm_expanded_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/prefix_codes_icd10cm_expanded.csv")
prefix_codes_icd10cm_expanded_df

,CUI,CODE,STR,PREFIX_CODE
0,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A00
1,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A01
2,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A02
3,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A03
4,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A04
...,...,...,...,...
5407,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z95
5408,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z96
5409,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z97
5410,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z98


## GenAI Wrapper Functions

In [25]:
"""Code for interacting with GenAI prompts"""


def load_multiple_objects_into_model(objects_dict, prompt, model="models/gemini-2.5-flash",
                                    count_tokens=None,
                                    serialization="json", structured_responses=False):
  """
    Loads multiple object into a generative model. Objects are passed in a dictionary
    where the key is used in the prompt before the value of the object.
  """
  model_obj = genai.GenerativeModel(model)

  object_prompt = ""
  for key in objects_dict:
    object_prompt += f"{key} - Object serialzed in '{serialization}'\n"
    if serialization == "json":
      object_prompt += json.dumps(objects_dict[key]) + "\n\n"
    else:
      object_prompt += str(objects_dict[key]) + "\n\n"

  full_prompt = f"{object_prompt}\n\n{prompt}"

  if count_tokens:
    response = model_obj.count_tokens(full_prompt)
    print(f"Total prompt size: {len(full_prompt)} which translates into {response}")
    return response
  else:
    response = model_obj.generate_content(full_prompt)

    if structured_responses:
      try:
          return json.loads(response.text)
      except json.JSONDecodeError:

        return json.loads(response.text.split("```")[1][5:])
      except json.DecoderError:
        print(response.text.split("```")[1])
        raise RuntimeError("JSON Decoder Error")
    else:
      return response.text

def evaluate_code(code, source="ICD10CM", model="models/gemini-2.5-flash", prompt="Can you summarize how the code should be used?",
                  token_size_only=False, structured_responses=False):
  """Gets context for a UMLS sourced code and evaluates a prompt against it"""

  return load_multiple_objects_into_model(model=model, objects_dict={source: get_code_source_information(code, source)},
                                  prompt=prompt, structured_responses=structured_responses,
                                  count_tokens=token_size_only)




## GenAI Examples

### CCSR examples

In [178]:
import io
from re import split
import pprint
def ccsr_to_ICD10CM(overall_ccsr_criteria, filter_icd10cm_criteria, output_criteria="Return ICD10CM codes with text descriptions in a CSV file format and quote all text fields with "".  The header for CSV file should be the columns of ""code,description"". Explain why certain codes were exlcuded from the list."):
  ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}

  prompt_1 = overall_ccsr_criteria + "Only return CCSR code as elements in a JSON list."

  ccsr_codes = load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt=prompt_1, structured_responses=True)

  print(f"Found the following CCSR codes: {ccsr_codes}")
  print("Description of CCSR Codes:")
  pprint.pprint(
  {pair["code"]: pair["description"] for pair in ccsr_dict["CCSR"] if pair["code"] in ccsr_codes})

  print("")

  ccsr_umls_dict = {c: get_code_source_information(c, "CCSR_ICD10CM") for c in ccsr_codes}

  prompt_2 = filter_icd10cm_criteria + output_criteria

  return load_multiple_objects_into_model(objects_dict=ccsr_umls_dict, prompt=prompt_2, structured_responses=False)

def extract_csv_codes_to_df(output):
  split_output = output.split("```")
  if len(split_output) == 3:
    if split_output[1][0:3] == "csv":
      StringIO = io.StringIO(split_output[1][3:])
      df = pd.read_csv(StringIO)
      columns = df.columns
      df.columns = [c.lower() for c in columns]
      return df.drop_duplicates()
    else:
      return None
  else:
    return None


In [179]:
ccsr_codes = ccsr_to_ICD10CM("Find CCSR codes related to diabetes", "Return ICD10CM codes that are associated with complications to vision.")
print(ccsr_codes)

Found the following CCSR codes: ['END002', 'END003', 'END004', 'END005', 'END006', 'PRG019', 'SYM018']
Description of CCSR Codes:
{'END002': 'Diabetes mellitus without complication',
 'END003': 'Diabetes mellitus with complication',
 'END004': 'Diabetes mellitus, Type 1',
 'END005': 'Diabetes mellitus, Type 2',
 'END006': 'Diabetes mellitus, due to underlying condition, drug or chemical '
           'induced, or other specified type',
 'PRG019': 'Diabetes or abnormal glucose tolerance complicating pregnancy; '
           'childbirth; or the puerperium',
 'SYM018': 'Prediabetes'}

```csv
code,description
"E08.321","Diabetes mellitus due to underlying condition with mild nonproliferative diabetic retinopathy with macular edema"
"E08.329","Diabetes mellitus due to underlying condition with mild nonproliferative diabetic retinopathy without macular edema"
"E08.349","Diabetes mellitus due to underlying condition with severe nonproliferative diabetic retinopathy without macular edema"
"E08.3

In [180]:
codes_df = extract_csv_codes_to_df(ccsr_codes)
codes_df.drop_duplicates()

,code,description
0,E08.321,Diabetes mellitus due to underlying condition ...
1,E08.329,Diabetes mellitus due to underlying condition ...
2,E08.349,Diabetes mellitus due to underlying condition ...
3,E08.359,Diabetes mellitus due to underlying condition ...
4,E08.339,Diabetes mellitus due to underlying condition ...
...,...,...
301,E13.3593,Other specified diabetes mellitus with prolife...
302,E13.37X1,Other specified diabetes mellitus with diabeti...
303,E13.37X2,Other specified diabetes mellitus with diabeti...
304,E13.37X3,Other specified diabetes mellitus with diabeti...


In [192]:
merged_codes_df = codes_df.merge(full_icd10cm_df, left_on="code", right_on="concept_code")[["concept_id", "code", "description", "domain_id"]].drop_duplicates().sort_values("code")
merged_codes_df

,concept_id,code,description,domain_id
10,45533010,E08.311,Diabetes mellitus due to underlying condition ...,Condition
7,45552373,E08.319,Diabetes mellitus due to underlying condition ...,Condition
0,45600633,E08.321,Diabetes mellitus due to underlying condition ...,Condition
12,37200028,E08.3212,Diabetes mellitus due to underlying condition ...,Condition
19,37200029,E08.3213,Diabetes mellitus due to underlying condition ...,Condition
...,...,...,...,...
302,37200308,E13.37X1,Other specified diabetes mellitus with diabeti...,Condition
303,37200309,E13.37X2,Other specified diabetes mellitus with diabeti...,Condition
304,37200310,E13.37X3,Other specified diabetes mellitus with diabeti...,Condition
305,37200311,E13.37X9,Other specified diabetes mellitus with diabeti...,Condition


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['concept_id'].plot(kind='hist', bins=20, title='concept_id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2.plot(kind='scatter', x='index', y='concept_id', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_4['concept_id'].plot(kind='line', figsize=(8, 4), title='concept_id')
plt.gca().spines[['top', 'right']].set_visible(False)

In [193]:
codes_df[~codes_df["code"].isin(merged_codes_df["code"])].sort_values("code")

,code,description


In [94]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt="Can you summarize which CCSR code for diabetes or complications from diabetes?"))

Based on the provided CCSR codes, here are the codes related to diabetes or complications from diabetes:

*   **END002**: Diabetes mellitus without complication
*   **END003**: Diabetes mellitus with complication
*   **END004**: Diabetes mellitus, Type 1
*   **END005**: Diabetes mellitus, Type 2
*   **END006**: Diabetes mellitus, due to underlying condition, drug or chemical induced, or other specified type
*   **PRG019**: Diabetes or abnormal glucose tolerance complicating pregnancy; childbirth; or the puerperium
*   **SYM018**: Prediabetes


In [95]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
code_prompt="Can you summarize which CCSR code for diabetes or complications from diabetes? Only return the CCSR codes as JSON list."
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt=code_prompt, structured_responses=True))

['END002', 'END003', 'END004', 'END005', 'END006', 'PRG019', 'SYM018']


In [28]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
print(load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_dict, prompt="Can you summarize which CCSR for breast cancer?"))

Based on the provided CCSR data, the codes for breast cancer are:

*   **NEO029**: Breast cancer - ductal carcinoma in situ (DCIS)
*   **NEO030**: Breast cancer - all other types


In [29]:
prompt="Can you find which CCSR code are related to breast cancer. Return the CCSR codes only in a JSON list"
ccsr_codes = load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_dict, prompt=prompt, structured_responses=True)
ccsr_codes

['NEO029', 'NEO030']

In [31]:
ccsr_umls_dict = {c: get_code_source_information(c, "CCSR_ICD10CM") for c in ccsr_codes}
print(load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_umls_dict,
                                      prompt="Return ICD10CM codes and descriptions in a CSV format where primary tumor occurred in the breast. Explain why certain codes were excluded"))

Here are the ICD-10-CM codes and descriptions for primary breast tumors in CSV format, followed by an explanation of excluded codes:

```csv
Code,Description
C50.119,Malignant neoplasm of central portion of unspecified female breast
C50.011,Malignant neoplasm of nipple and areola, right female breast
C50.021,Malignant neoplasm of nipple and areola, right male breast
C50.111,Malignant neoplasm of central portion of right female breast
C50.029,Malignant neoplasm of nipple and areola, unspecified male breast
C50.112,Malignant neoplasm of central portion of left female breast
C50.012,Malignant neoplasm of nipple and areola, left female breast
C50.019,Malignant neoplasm of nipple and areola, unspecified female breast
C50.022,Malignant neoplasm of nipple and areola, left male breast
C50.122,Malignant neoplasm of central portion of left male breast
C50.129,Malignant neoplasm of central portion of unspecified male breast
C50.211,Malignant neoplasm of upper-inner quadrant of right female breast

In [ ]:
import io
df = pd.read_csv(io.StringIO(evaluate_code("END004", "CCSR_ICD10CM", prompt='Return a CSV output (escape text fields with "") of ICD10CM codes and descriptions that are associated with complications to the eyes')))
df

,ICD10CM Code,Description
0,E10.331,Type 1 diabetes mellitus with moderate nonprol...
1,E10.341,Type 1 diabetes mellitus with severe nonprolif...
2,E10.311,Type 1 diabetes mellitus with unspecified diab...
3,E10.321,Type 1 diabetes mellitus with mild nonprolifer...
4,E10.339,Type 1 diabetes mellitus with moderate nonprol...
...,...,...
59,E10.37X1,Type 1 diabetes mellitus with diabetic macular...
60,E10.37X2,Type 1 diabetes mellitus with diabetic macular...
61,E10.3599,Type 1 diabetes mellitus with proliferative di...
62,E10.37X9,Type 1 diabetes mellitus with diabetic macular...


In [33]:
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt="Can you summarize which CCSR code for diabetes or complications from diabetes?"))

Based on the provided CCSR codes, here are the codes related to diabetes or complications from diabetes:

*   **END002:** Diabetes mellitus without complication
*   **END003:** Diabetes mellitus with complication
*   **END004:** Diabetes mellitus, Type 1
*   **END005:** Diabetes mellitus, Type 2
*   **END006:** Diabetes mellitus, due to underlying condition, drug or chemical induced, or other specified type
*   **PRG019:** Diabetes or abnormal glucose tolerance complicating pregnancy; childbirth; or the puerperium
*   **SYM018:** Prediabetes


In [48]:
print(evaluate_code("INJ031", "CCSR_ICD10CM"))

This JSON object defines a specific **CCSR (Clinical Classifications Software Refined) category** within the **ICD-10-CM vocabulary** related to "Allergic reactions".

Here's how the code should be used, based on its structure and content:

1.  **Identify the CCSR Category:**
    *   `"code": "INJ031"`: This is the unique identifier for this specific CCSR category.
    *   `"name": "Allergic reactions"`: This is the human-readable name of the category.
    *   `"vocabulary": "CCSR_ICD10CM"`: Confirms it belongs to the CCSR classification system for ICD-10-CM diagnoses.

2.  **Understand the Concept/Definition:**
    *   The `"concepts"` block provides formal definitions (from SNOMEDCT_US and NCI) for what "Allergic reactions" entails clinically. This ensures a consistent understanding of the category's scope.
    *   `"concept_uri"`: Allows for external lookup of the CUI (Concept Unique Identifier) `C1527304` for more detailed information if needed.

3.  **Map Specific ICD-10-CM Codes 

In [52]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are related to the ear."))

```csv
ICD10CM Code,Description
H65.111,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), right ear
H65.113,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), bilateral
H65.114,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, right ear
H65.119,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), unspecified ear
H65.115,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, left ear
H65.412,Chronic allergic otitis media, left ear
H65.413,Chronic allergic otitis media, bilateral
H65.116,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, bilateral
H65.411,Chronic allergic otitis media, right ear
H65.117,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, unspecified ear
H65.112,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), left ear
H65.419,Chronic allergic otitis media, u

In [53]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are related to the skin."))

```csv
ICD10CM Code,Description
H01.112,Allergic dermatitis of right lower eyelid
H01.111,Allergic dermatitis of right upper eyelid
H01.113,Allergic dermatitis of right eye, unspecified eyelid
H01.114,Allergic dermatitis of left upper eyelid
H01.119,Allergic dermatitis of unspecified eye, unspecified eyelid
H01.131,Eczematous dermatitis of right upper eyelid
H01.132,Eczematous dermatitis of right lower eyelid
H01.135,Eczematous dermatitis of left lower eyelid
H01.136,Eczematous dermatitis of left eye, unspecified eyelid
H01.139,Eczematous dermatitis of unspecified eye, unspecified eyelid
H01.134,Eczematous dermatitis of left upper eyelid
H01.115,Allergic dermatitis of left lower eyelid
H01.116,Allergic dermatitis of left eye, unspecified eyelid
H01.133,Eczematous dermatitis of right eye, unspecified eyelid
L27.0,Generalized skin eruption due to drugs and medicaments taken internally
L20.83,Infantile (acute) (chronic) eczema
L23.81,Allergic contact dermatitis due to animal (cat) (dog) d

In [54]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="How many ICD10CM codes are in this group?"))

There are **80** unique ICD10CM codes in this group.


In [55]:
len(get_code_source_information("INJ031", "CCSR_ICD10CM")["relationships"])

138

In [56]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of the ICD10CM codes and descriptions and drop duplicate rows"))

```csv
ICD10CM Code,Description
K90.41,Non-celiac gluten sensitivity
H01.112,Allergic dermatitis of right lower eyelid
H01.111,Allergic dermatitis of right upper eyelid
H01.113,Allergic dermatitis of right eye, unspecified eyelid
H01.114,Allergic dermatitis of left upper eyelid
H01.119,Allergic dermatitis of unspecified eye, unspecified eyelid
H01.131,Eczematous dermatitis of right upper eyelid
H01.132,Eczematous dermatitis of right lower eyelid
H01.135,Eczematous dermatitis of left lower eyelid
H01.136,Eczematous dermatitis of left eye, unspecified eyelid
H01.139,Eczematous dermatitis of unspecified eye, unspecified eyelid
H01.134,Eczematous dermatitis of left upper eyelid
H01.115,Allergic dermatitis of left lower eyelid
H01.116,Allergic dermatitis of left eye, unspecified eyelid
H01.133,Eczematous dermatitis of right eye, unspecified eyelid
H65.111,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), right ear
H65.113,Acute and subacute allergic otitis media (muco

In [57]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes that are associated with gestational diabetes"))

The provided JSON object describes "Diabetes mellitus, Type 1" (code: END004) and its related ICD-10-CM codes. It **does not contain any ICD-10-CM codes that are specifically associated with gestational diabetes**.

The codes listed in the `relationships` section are all classifications or specific manifestations of Type 1 diabetes, including when Type 1 diabetes pre-exists in pregnancy.


In [88]:
print(evaluate_code("PREG", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes that are associated with gestational diabetes"))

Here are the ICD-10-CM codes associated with gestational diabetes, covering different stages of pregnancy, childbirth, and the puerperium, and various control types:

O24.410,O24.414,O24.415,O24.419,O24.420,O24.424,O24.425,O24.429,O24.430,O24.434,O24.435,O24.439,Z86.3A


In [62]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are associated with pregnancy"))

```csv
ICD10CM_Code,Description
O24.013,Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester
O24.012,Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester
O24.011,Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester
O24.02,Pre-existing type 1 diabetes mellitus, in childbirth
O24.03,Pre-existing type 1 diabetes mellitus, in the puerperium
O24.019,Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester
```


In [63]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a JSON document of ICD10CM codes and descriptions that are associated with pregnancy and complications"))

```json
[
  {
    "code": "O24.013",
    "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester"
  },
  {
    "code": "O24.012",
    "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester"
  },
  {
    "code": "O24.011",
    "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester"
  },
  {
    "code": "O24.02",
    "description": "Pre-existing type 1 diabetes mellitus, in childbirth"
  },
  {
    "code": "O24.03",
    "description": "Pre-existing type 1 diabetes mellitus, in the puerperium"
  },
  {
    "code": "O24.019",
    "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester"
  }
]
```


In [64]:
print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt="Return a CSV file of ICD10CM codes and descriptions related to pre-exisiting type 1 and/or type 2 diabetes. After generating the list explain why certain codes were excluded"))

```csv
ICD10CM Code,Description
"O24.011","Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester"
"O24.012","Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester"
"O24.013","Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester"
"O24.019","Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester"
"O24.02","Pre-existing type 1 diabetes mellitus, in childbirth"
"O24.03","Pre-existing type 1 diabetes mellitus, in the puerperium"
"O24.111","Pre-existing type 2 diabetes mellitus, in pregnancy, first trimester"
"O24.112","Pre-existing type 2 diabetes mellitus, in pregnancy, second trimester"
"O24.113","Pre-existing type 2 diabetes mellitus, in pregnancy, third trimester"
"O24.119","Pre-existing type 2 diabetes mellitus, in pregnancy, unspecified trimester"
"O24.12","Pre-existing type 2 diabetes mellitus, in childbirth"
"O24.13","Pre-existing type 2 diabetes mellitus, in the puerperium"
```

**Explanation of excluded codes:**

T

In [65]:
prompt="""Return a comma separated list of all the ICD10CM codes for type 1 or type 2 diabetes during pregnancy.
Codes in the list should be single quoted. Explain after listing the ICD10CM codes why certain codes
from the list were excluded."""

print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt=prompt))

The ICD-10-CM codes for type 1 or type 2 diabetes during pregnancy are:
'O24.011', 'O24.012', 'O24.013', 'O24.019', 'O24.111', 'O24.112', 'O24.113', 'O24.119'

The following categories of codes were excluded from the list:

*   **Codes for "childbirth" or "puerperium"**: The request specifically asked for codes "during pregnancy." Therefore, codes that describe diabetes complicating "childbirth" (e.g., O24.02, O24.12) or the "puerperium" (e.g., O24.03, O24.13) were excluded, as these represent different phases of the obstetric period.
*   **Codes for "Gestational diabetes mellitus" (O24.4xx)**: The request explicitly specified "type 1 or type 2 diabetes," and gestational diabetes is a distinct clinical entity.
*   **Codes for "Unspecified pre-existing diabetes mellitus" (O24.3xx)**: These codes do not specify whether the pre-existing diabetes is type 1 or type 2.
*   **Codes for "Other pre-existing diabetes mellitus" (O24.8xx)**: These codes refer to forms of diabetes other than type 1

In [66]:
print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt="When should I used this code?"))

The CCSR (Clinical Classifications Software Refined) code **PRG019** should be used when classifying a patient's condition related to **diabetes or abnormal glucose tolerance that is complicating pregnancy, childbirth, or the puerperium (the period immediately following childbirth).**

This is a broad classification code designed to group various specific ICD-10-CM codes under a common category for easier data analysis and reporting.

**You would use this code to represent scenarios such as:**

*   **Pre-existing Diabetes (Type 1 or Type 2) Complicating:**
    *   Any trimester of pregnancy.
    *   Childbirth.
    *   The postpartum period (puerperium).
    *   *Examples of specific ICD-10-CM codes grouped under PRG019:* O24.011 (Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester), O24.12 (Pre-existing type 2 diabetes mellitus, in childbirth), O24.03 (Pre-existing type 1 diabetes mellitus, in the puerperium).

*   **Gestational Diabetes Mellitus (GDM) Complicating:**

### ICD10CM

In [84]:
full_icd10cm_df[["concept_code", "STR", "concept_id"]][full_icd10cm_df["TTY"] == "PT"]

,concept_code,STR,concept_id
0,O02.1,Missed abortion,35209522
1,L73.0,Acne keloid,35208638
2,M95.4,Acquired deformity of chest and rib,35209145
3,F43.24,Adjustment disorder with disturbance of conduct,45600747
4,R48.0,Dyslexia and alexia,35211378
...,...,...,...
74255,Z68.56,"Body mass index [BMI] pediatric, greater than ...",1102847
74256,Z83.72,Family history of familial adenomatous polyposis,1102848
74257,Z67.A2,Duffy a positive,1102825
74258,F50.84,Rumination disorder in adults,1102672


In [85]:
icd10cm_pt_dict = full_icd10cm_df[["concept_code", "STR"]][full_icd10cm_df["TTY"] == "PT"].sort_values("concept_code").to_dict(orient="records")

In [86]:
load_multiple_objects_into_model(objects_dict={"ICD10CM": icd10cm_pt_dict}, prompt="Find all ICD10CM codes for cancer originating in breast tissue. List results in CSV format and include the description and concept_id", count_tokens=True)

Total prompt size: 8635788 which translates into total_tokens: 2497125



total_tokens: 2497125

In [87]:
t = full_icd10cm_df[["concept_code", "STR"]][full_icd10cm_df["TTY"] == "PT"].sort_values("concept_code").to_csv(index=False)

In [72]:
load_multiple_objects_into_model(objects_dict={"ICD10CM (CSV format)": t}, count_tokens=True, prompt="")

Total prompt size: 6658605 which translates into total_tokens: 1797773



total_tokens: 1797773

### Elixhauser comorbidities

In [73]:
elixhauser_codes_map_snomed_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/elixhauser_codes_map_snomed.csv")
elixhauser_codes_map_snomed_df

,code,code_type,description,concept_id,code_class,concept_name,domain_id,concept_code,domain_id.1,concept_class_id,vocabulary_id,mapped_concept_id,mapped_concept_name,mapped_domain_id,mapped_concept_code,mapped_concept_class_id,mapped_vocabulary_id
0,B20,ICD10CM,Human immunodeficiency virus [HIV] disease,35205776,aids,Human immunodeficiency virus [HIV] disease,Condition,B20,Condition,3-char billing code,ICD10CM,439727,Human immunodeficiency virus infection,Condition,86406008,Disorder,SNOMED
1,E52,ICD10CM,Niacin deficiency [pellagra],35206985,alcohol,Niacin deficiency [pellagra],Condition,E52,Condition,3-char billing code,ICD10CM,4303664,Niacin deficiency,Condition,418279001,Disorder,SNOMED
2,F10.10,ICD10CM,"Alcohol abuse, uncomplicated",45595845,alcohol,"Alcohol abuse, uncomplicated",Condition,F10.10,Condition,5-char billing code,ICD10CM,433753,Alcohol abuse,Condition,15167005,Disorder,SNOMED
3,F10.11,ICD10CM,"Alcohol abuse, in remission",1326497,alcohol,"Alcohol abuse, in remission",Condition,F10.11,Condition,5-char billing code,ICD10CM,433753,Alcohol abuse,Condition,15167005,Disorder,SNOMED
4,F10.11,ICD10CM,"Alcohol abuse, in remission",1326497,alcohol,"Alcohol abuse, in remission",Condition,F10.11,Condition,5-char billing code,ICD10CM,35622958,Disorder in remission,Condition,765205004,Disorder,SNOMED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4481,E44.1,ICD10CM,Mild protein-calorie malnutrition,35206969,wloss,Mild protein-calorie malnutrition,Condition,E44.1,Condition,4-char billing code,ICD10CM,4096196,Mild protein-calorie malnutrition (weight for ...,Condition,190603003,Disorder,SNOMED
4482,E45,ICD10CM,Retarded development following protein-calorie...,35206970,wloss,Retarded development following protein-calorie...,Condition,E45,Condition,3-char billing code,ICD10CM,435773,Arrested development following protein-calorie...,Condition,74257000,Disorder,SNOMED
4483,E46,ICD10CM,Unspecified protein-calorie malnutrition,35206971,wloss,Unspecified protein-calorie malnutrition,Condition,E46,Condition,3-char billing code,ICD10CM,4276360,Undernutrition,Condition,65404009,Disorder,SNOMED
4484,R63.4,ICD10CM,Abnormal weight loss,35211409,wloss,Abnormal weight loss,Observation,R63.4,Observation,4-char billing code,ICD10CM,435928,Abnormal weight loss,Observation,267024001,Clinical Finding,SNOMED


In [74]:
abridged_elixhauser_codes_map_snomed_df = elixhauser_codes_map_snomed_df[["code_class","concept_id","code", "concept_name", "mapped_concept_id", "mapped_vocabulary_id", "mapped_concept_code", "mapped_concept_name"]]
abridged_elixhauser_codes_map_snomed_df

,code_class,concept_id,code,concept_name,mapped_concept_id,mapped_vocabulary_id,mapped_concept_code,mapped_concept_name
0,aids,35205776,B20,Human immunodeficiency virus [HIV] disease,439727,SNOMED,86406008,Human immunodeficiency virus infection
1,alcohol,35206985,E52,Niacin deficiency [pellagra],4303664,SNOMED,418279001,Niacin deficiency
2,alcohol,45595845,F10.10,"Alcohol abuse, uncomplicated",433753,SNOMED,15167005,Alcohol abuse
3,alcohol,1326497,F10.11,"Alcohol abuse, in remission",433753,SNOMED,15167005,Alcohol abuse
4,alcohol,1326497,F10.11,"Alcohol abuse, in remission",35622958,SNOMED,765205004,Disorder in remission
...,...,...,...,...,...,...,...,...
4481,wloss,35206969,E44.1,Mild protein-calorie malnutrition,4096196,SNOMED,190603003,Mild protein-calorie malnutrition (weight for ...
4482,wloss,35206970,E45,Retarded development following protein-calorie...,435773,SNOMED,74257000,Arrested development following protein-calorie...
4483,wloss,35206971,E46,Unspecified protein-calorie malnutrition,4276360,SNOMED,65404009,Undernutrition
4484,wloss,35211409,R63.4,Abnormal weight loss,435928,SNOMED,267024001,Abnormal weight loss


In [75]:
ab_al_dict = abridged_elixhauser_codes_map_snomed_df[abridged_elixhauser_codes_map_snomed_df["code_class"] == "alcohol"].to_dict(orient="records")
len(ab_al_dict)

223

In [76]:
abridged_elixhauser_codes_map_snomed_df[abridged_elixhauser_codes_map_snomed_df["code_class"] == "alcohol"][["mapped_concept_id", "mapped_concept_code", "mapped_concept_name"]].drop_duplicates()

,mapped_concept_id,mapped_concept_code,mapped_concept_name
1,4303664,418279001,Niacin deficiency
2,433753,15167005,Alcohol abuse
4,35622958,765205004,Disorder in remission
6,4104431,25702006,Alcohol intoxication
8,4088373,18653004,Alcohol intoxication delirium
11,375519,191480000,Alcohol withdrawal
13,377830,8635005,Alcohol withdrawal delirium
16,37165554,1255066009,Perceptual disturbance due to alcohol withdrawal
20,37164789,1234779007,Mood disorder caused by ethanol
22,442582,61144001,Alcohol-induced psychotic disorder with delusions


In [77]:
print(load_multiple_objects_into_model(objects_dict={"ICD10CM mapped to SNOMED": ab_al_dict}, count_tokens=False, prompt="Generate a CSV list mapped codes with descriptions mapped concept_id that are not related to alcohol or effects of alcohol"))

```csv
Mapped Code,Mapped Description
218249005,Disorder due to and following accidental poisoning
219174008,Late effect of self inflicted injury
219365001,Late effect of injury of unknown intent
35622958,Disorder in remission
35625043,Late effect due to homicide attempt
418279001,Niacin deficiency
440279,Accident
439235,Self inflicted injury
```


In [78]:
elixhauser_codes_map_snomed_df["code_class"].drop_duplicates()

,code_class
0,aids
1,alcohol
224,blane
225,carit
297,chf
345,coag
372,cpd
442,dane
459,depre
492,diabc


In [79]:
def reason_over_icd10_codes_mapped(code_class="wloss", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=""):
  mapped_dict = snomed_code_mapped_df[snomed_code_mapped_df["code_class"] == code_class].to_dict(orient="records")
  return load_multiple_objects_into_model(objects_dict={"ICD10CM mapped to SNOMED": mapped_dict}, count_tokens=False, prompt=prompt)

In [80]:
prompt="Generate a CSV list mapped SNOMED codes with descriptions mapped concept_ids that are not related to weight loss"
print(reason_over_icd10_codes_mapped(code_class="wloss", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=prompt))

It appears that all the provided ICD10CM codes in your JSON data have a `code_class` of "wloss" (weight loss). Therefore, if we filter for SNOMED codes *not* related to weight loss, the list will be empty based on your current data and the `code_class` attribute.

Here's the CSV output, which will only contain the header:

```csv
SNOMED_Code,SNOMED_Description
```


In [81]:
prompt="""Generate a CSV list mapped SNOMED codes with descriptions mapped concept_ids
 and mapped_concept_codes that are not related to heart failure. Remove duplicates."""
print(reason_over_icd10_codes_mapped(code_class="chf", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=prompt))

```csv
mapped_concept_id,mapped_concept_name,mapped_concept_code
35207667,Rheumatic heart disease,23685000
4110961,Generalized ischemic myocardial dysfunction,194849004
4163710,Dilated cardiomyopathy,399020009
4190773,Restrictive cardiomyopathy,415295002
37164777,Alcoholic cardiomyopathy,1234750000
35615148,Cardiomyopathy caused by drug,16228931000119102
321319,Cardiomyopathy,85898001
320746,Cardiomyopathy associated with another disorder,195029002
```


### Rxnorm

In [ ]:
print(evaluate_code("153165", "RXNORM"))

This JSON object represents an entry for the drug **"Lipitor"** within the **RXNORM** vocabulary. It provides a comprehensive set of information about this drug, particularly focusing on its identity, conceptual links, and various relationships with other related entities (like active ingredients, dosage forms, and synonyms).

Here's a breakdown of how this code should be used, field by field:

1.  **`code` (e.g., "153165")**:
    *   **Usage**: This is the unique identifier for "Lipitor" within the RXNORM vocabulary. It should be used for precise lookups, referencing, and linking to this specific concept in other systems or databases that utilize RXNORM.

2.  **`name` (e.g., "Lipitor")**:
    *   **Usage**: This is the primary human-readable name for the concept. It's typically used for display in user interfaces, search results, or reports where a common name is needed.

3.  **`vocabulary` (e.g., "RXNORM")**:
    *   **Usage**: Confirms the source vocabulary for the entry. Useful whe

In [ ]:
print(evaluate_code("83367", "RXNORM", prompt="list all contrandications for this medication"))

Based on the provided JSON, here are the contraindications for atorvastatin:

*   Drug Hypersensitivity
*   Liver Failure
*   Pregnancy
*   Lactation
*   Liver Diseases


### CPT

In [ ]:
print(evaluate_code("33361", "CPT"))

Based on the provided JSON object, here's a summary of how CPT code 33361 should be used:

**CPT Code 33361: Transcatheter Aortic Valve Replacement (TAVR/TAVI) via Percutaneous Femoral Artery Approach**

This code describes a specific type of transcatheter aortic valve replacement (TAVR or TAVI) procedure.

**Key Usage Points:**

1.  **Procedure:** It covers the replacement of the aortic valve using a prosthetic valve, performed via a **transcatheter approach** (meaning through blood vessels, not open surgery) specifically through the **percutaneous femoral artery** (an incision through the skin into the femoral artery in the groin).

2.  **Specialties:** This procedure is typically performed by specialists in:
    *   Internal Medicine (Cardiovascular Disease, Interventional Cardiology)
    *   Thoracic Surgery (Cardiothoracic Vascular Surgery)

3.  **Add-on Codes (Codes to be billed *in addition to* 33361 when applicable):**
    *   **Cardiopulmonary Bypass Support:** If cardiopulmon

### LOINC

In [ ]:
print(evaluate_code("4548-4", "LNC"))

The LOINC code `4548-4` represents a quantitative laboratory test used to measure the **Hemoglobin A1c (HbA1c) as a mass fraction of total Hemoglobin** in a blood sample collected at a specific **point in time**.

Here's a breakdown of how this code should be used and its meaning:

*   **`Hemoglobin A1c/Hemoglobin.total` (Component):** This is the substance or entity being measured. It specifically refers to the ratio of glycated hemoglobin (HbA1c) to the total hemoglobin in the blood. HbA1c reflects average blood glucose levels over the past 2-3 months.
*   **`MFr` (Property):** This stands for "Mass Fraction." It indicates that the measurement is expressed as a ratio of masses (e.g., a percentage of total hemoglobin).
*   **`Pt` (Time Aspect):** This denotes "Point in time," meaning the measurement is taken at a single, specific moment, not over a duration.
*   **`Bld` (System):** This specifies the sample type, which is "Blood," and more specifically, "Whole blood" as indicated in t

### HPO

In [92]:
print(evaluate_code("HP:0033630", "HPO"))

The provided JSON object is a serialized representation of a single entry from the **Human Phenotype Ontology (HPO)**. It details the HPO term **"Brain fog" (HP:0033630)** and its associated metadata.

Here's a summary of how this JSON data should be used:

1.  **Understanding a Specific Phenotype:**
    *   **Identify the Term:** The `code` (`HP:0033630`) and `name` (`Brain fog`) immediately tell you which HPO term is being described.
    *   **Access Definitions:** The `concepts` section, specifically `concepts.C0015676.definitions`, provides formal definitions from sources like MSH (MeSH) and HPO itself. This is crucial for understanding the precise meaning of "Brain fog" within a clinical or research context.
    *   **Get Context/Comments:** The `attributes.HPO_COMMENT` provides additional explanatory notes, examples of diseases where it's reported, or specific nuances of the term.
    *   **Determine Semantic Type:** `concepts.C0015676.semantic_type` (`Mental or Behavioral Dysfun

### OMIM

In [ ]:
print(evaluate_code("190070", "OMIM"))

This JSON object provides a structured and comprehensive representation of an **OMIM (Online Mendelian Inheritance in Man) entry** for a specific gene: **KRAS PROTOONCOGENE, GTPase (OMIM code 190070)**.

It is designed to be programmatically consumed and used for:

1.  **Core Identification and Metadata Retrieval:**
    *   `"code"`: Use `"190070"` as the primary unique identifier for this OMIM entry.
    *   `"name"`: Get the official human-readable name of the gene/entry (`"KRAS PROTOONCOGENE, GTPase"`).
    *   `"vocabulary"`: Confirms that the data originates from OMIM.

2.  **Semantic Enrichment and Concept Mapping (`concepts`):**
    *   This section links the OMIM entry to **Unified Medical Language System (UMLS) Concepts (CUIs)**.
    *   Each key (`"C3809005"`, `"C4016400"`, etc.) is a CUI, and its value contains:
        *   `"concept_uri"`: A direct link to the concept's details in the UMLS Terminology Services (UTS).
        *   `"semantic_type"`: Categorizes the concept (e

### SNOMED

In [ ]:
print(evaluate_code("44054006", "SNOMEDCT_US"))

This JSON object provides a comprehensive, machine-readable representation of the SNOMED CT concept "Type 2 diabetes mellitus" (code: 44054006). It should be used to:

1.  **Identify and Reference:** The `code` (44054006), `name` ("Type 2 diabetes mellitus"), and `vocabulary` ("SNOMEDCT_US") provide a unique and standardized identifier for this specific clinical concept. This is fundamental for clinical documentation and data exchange.

2.  **Understand Clinical Meaning:** The `concepts` section, particularly the `definitions` from various authoritative sources (NCI, HPO, MEDLINEPLUS, CSP, MSH), offers detailed textual descriptions of the disease. This is valuable for educational purposes, clinical interpretation, and ensuring a shared understanding of the term.

3.  **Navigate Hierarchical Relationships:** The `parents`, `children`, `ancestors`, and `descendants` arrays allow for traversing the SNOMED CT hierarchy. For example, you can see that "Type 2 diabetes mellitus" is a specific